# Chapter 6 — Distillation: Teacher → Student

**Goal:** Transfer knowledge from a large teacher model (DeepSeek) to a small student model (SmolLM) using pure distillation — the teacher generates ALL training data.

This chapter covers two forms of distillation:
1. **Hard-label distillation** — train the student on the teacher's text outputs
2. **Soft-label distillation** — train on the teacher's token-level probability distribution (richer signal)

We use few-shot seeding: give the teacher 4-5 hand-picked examples per class so it understands our labeling conventions, then have it generate the full dataset.

## 6.1 Setup & Few-Shot Seed Examples

These 16 examples teach DeepSeek our exact labeling style. The teacher will generate hundreds more following this pattern.

In [ ]:
import torch, json, os, time
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from openai import OpenAI
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "HuggingFaceTB/SmolLM-135M"

# Few-shot seed examples — teach the teacher our labeling style
SEED_EXAMPLES = [
    {"text": "app crashes when I tap the search button", "label": "bug"},
    {"text": "screen goes blank after updating to latest version", "label": "bug"},
    {"text": "notifications not showing up on my Android phone", "label": "bug"},
    {"text": "upload fails with error 500 on the checkout screen", "label": "bug"},

    {"text": "please add dark mode for the dashboard", "label": "feature_request"},
    {"text": "would love a bulk delete option in the editor", "label": "feature_request"},
    {"text": "wish I could set custom swipe gestures in the app", "label": "feature_request"},
    {"text": "can you add a built-in QR scanner", "label": "feature_request"},

    {"text": "this app is amazing, so intuitive to use", "label": "praise"},
    {"text": "love the new update, everything runs so smooth", "label": "praise"},
    {"text": "best productivity tool, recommended it to my team", "label": "praise"},
    {"text": "customer support was incredibly helpful today", "label": "praise"},

    {"text": "how do I change my account email address", "label": "question"},
    {"text": "does the free tier include cloud backup", "label": "question"},
    {"text": "where can I find the billing history", "label": "question"},
    {"text": "is there a storage limit per user", "label": "question"},
]

api_key = os.environ.get("DEEPSEEK_API_KEY")
if not api_key:
    raise RuntimeError("Set DEEPSEEK_API_KEY environment variable first")

client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
print("DeepSeek client ready")

## 6.2 Generate Training Data (Pure Teacher Distillation)

DeepSeek generates ALL training examples. The few-shot seeds in the prompt ensure consistent labeling. Target: 50 batches = ~500 examples.

In [ ]:
# Build few-shot prompt from seed examples
seed_text = "\n".join(
    f'  {{"text": "{ex["text"]}", "label": "{ex["label"]}"}}'
    for ex in SEED_EXAMPLES
)

SYSTEM_PROMPT = f"""\
You are a data labeling assistant. Classify user messages into exactly one of 4 categories:

- bug: something is broken, crashing, not working, producing errors, or malfunctioning
- feature_request: asking to ADD or CREATE new functionality that doesn't exist yet
- praise: compliment, positive feedback, appreciation, or thanks
- question: asking HOW something works, WHAT a policy is, WHERE to find something, or WHETHER something exists

Here are examples of correct labeling:
{seed_text}

CRITICAL: Follow the EXACT same labeling style as the examples above.
Return ONLY a JSON array of objects. No markdown, no explanation, no other text."""

all_teacher_data = []
BATCHES = 50

for batch_num in range(BATCHES):
    raw = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Generate 10 new user messages with labels. Make them diverse — vary sentence structure, vocabulary, and app features mentioned. Include all 4 categories."},
        ],
        temperature=0.9,
        max_tokens=1024,
    ).choices[0].message.content

    try:
        batch = json.loads(raw)
        added = 0
        for item in batch:
            text = item.get("text", "").strip()
            label = item.get("label", "").strip().lower()
            if text and label in {"bug", "feature_request", "praise", "question"}:
                all_teacher_data.append({"text": text, "label": label})
                added += 1
        counts = dict(Counter(item["label"] for item in all_teacher_data))
        print(f"Batch {batch_num+1:2d}: +{added} → {len(all_teacher_data):3d} total  {counts}")
    except json.JSONDecodeError:
        print(f"Batch {batch_num+1:2d}: JSON parse failed — skipping")

    time.sleep(0.3)  # rate limit courtesy

print(f"\nGenerated {len(all_teacher_data)} teacher-labeled examples")
print(f"Class distribution: {dict(Counter(item['label'] for item in all_teacher_data))}")

## 6.3 Save & Format

Save the generated data and format it as prompt→completion pairs for training.

In [ ]:
# Save generated data
os.makedirs("./generated", exist_ok=True)
with open("./generated/classification.json", "w") as f:
    json.dump(all_teacher_data, f, indent=2)
print(f"Saved {len(all_teacher_data)} examples to ./generated/classification.json")

# Format as prompt→completion pairs
def make_pair(item):
    prompt = f"Classify: {item['text']}\nLabel:"
    completion = f" {item['label']}"
    return {"prompt": prompt, "completion": completion}

raw_data = [make_pair(item) for item in all_teacher_data]
np.random.seed(42)
np.random.shuffle(raw_data)
split = int(0.85 * len(raw_data))
train_data = raw_data[:split]
eval_data = raw_data[split:]
print(f"Train: {len(train_data)}  |  Eval: {len(eval_data)}")

## 6.4 Load Model & Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)

def tokenize(example):
    prompt_tokens = tokenizer(example["prompt"], truncation=True, max_length=128, padding=False)
    completion_tokens = tokenizer(example["completion"], truncation=True, max_length=16, padding=False)
    input_ids = prompt_tokens["input_ids"] + completion_tokens["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_tokens["input_ids"]) + completion_tokens["input_ids"] + [tokenizer.eos_token_id]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataset = Dataset.from_list(train_data).map(tokenize, remove_columns=["prompt", "completion"])
eval_dataset = Dataset.from_list(eval_data).map(tokenize, remove_columns=["prompt", "completion"])
print(f"Tokenized: {len(train_dataset)} train / {len(eval_dataset)} eval")

## 6.5 Train (Hard-Label Distillation)

In [ ]:
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora_config)

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(x["attention_mask"] + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
    return {"input_ids": torch.tensor(input_ids), "attention_mask": torch.tensor(attention_mask), "labels": torch.tensor(labels)}

training_args = TrainingArguments(
    output_dir="./chapters/06-distillation/checkpoint",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=(device == "cuda"),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=collate_fn)
print("Training student on pure teacher-generated data...")
trainer.train()

## 6.6 Evaluate

Same test set as Chapter 2 — direct comparison between hand-crafted (Ch2) and distilled (Ch6).

In [ ]:
VALID_LABELS = {"bug", "feature_request", "praise", "question"}

def classify(text):
    prompt = f"Classify: {text}\nLabel:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=3,
                                 do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id,
                                 eos_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Label:" in full:
        raw = full.split("Label:")[-1].strip()
        for label in VALID_LABELS:
            if raw.startswith(label):
                return label
    return raw.split()[0] if raw.split() else "???"

test_texts = [
    ("the app keeps crashing when I try to upload a photo", "bug"),
    ("would be great if we could have dark mode", "feature_request"),
    ("this is honestly the best note-taking app I've used", "praise"),
    ("how do I reset my password from the mobile app", "question"),
    ("getting a blank screen after the latest update", "bug"),
    ("please add markdown support to the editor", "feature_request"),
    ("the onboarding was so smooth, loved every bit of it", "praise"),
    ("does the free version include cloud sync", "question"),
]

print("Distilled student predictions:\n")
correct = 0
for text, exp in test_texts:
    pred = classify(text)
    ok = "OK" if pred == exp else f"WRONG (expected {exp})"
    if pred == exp:
        correct += 1
    print(f"  [{ok}] → {pred}")
print(f"\nAccuracy: {correct}/{len(test_texts)} ({100*correct/len(test_texts):.0f}%)")

## 6.7 Results Comparison

In [ ]:
# Re-run evaluation to get live accuracy for the comparison table
correct = 0
for text, exp in test_texts:
    if classify(text) == exp:
        correct += 1
ch6_acc = 100 * correct / len(test_texts)

print("Comparison on the 8-item classification test set:\n")
print(f"  {'Method':<40} {'Accuracy':>10}")
print(f"  {'─'*40} {'─'*10}")
print(f"  {'Untrained SmolLM (random guess)':<40} {'~25%':>10}")
print(f"  {'Ch2: Fine-tuned (hand-crafted 400 ex)':<40} {'~88%':>10}")
print(f"  {'Ch6: Pure distillation (teacher 500 ex)':<40} {f'{ch6_acc:.0f}%':>10}")
print(f"  {'DeepSeek teacher itself':<40} {'~98%':>10}")

print("\nKey takeaways:")
print("  1. Few-shot seeds → teacher learns YOUR labeling conventions")
print("  2. 500 teacher-generated examples → student approaches teacher quality")
print("  3. Same fine-tuning code as Ch2 — only the data source changed")
print("  4. Student (135M params, ~270MB) runs on laptop; teacher needs cloud GPU")

## 6.8 Soft-Label Distillation — The Concept

Hard labels only tell the student *what* the teacher said. Soft labels tell the student *how confident* the teacher was about every possible answer.

**Example:** For the input "the app keeps crashing when I upload", the teacher might output:

| Token | Hard label | Soft label (probability) |
|---|---|---|
| `bug` | 1.0 | 0.92 |
| `question` | 0.0 | 0.06 |
| `feature_request` | 0.0 | 0.02 |
| `praise` | 0.0 | 0.00 |

The soft label reveals that the teacher thinks "bug" is 92% likely — but "question" is also possible (6%). The student learns **class similarity**: messages about broken things and questions about broken things are more similar than messages praising the app.

This is what **temperature** controls: higher temperature → softer distribution → more information about token relationships.

## 6.9 Temperature Demo

In [ ]:
def apply_temperature(logits, T):
    scaled = np.array(logits) / T
    exp_scaled = np.exp(scaled - np.max(scaled))
    return exp_scaled / exp_scaled.sum()

teacher_logits = [5.0, 1.0, 0.2, -2.0]  # bug, feature_request, praise, question
labels = ["bug", "feature_request", "praise", "question"]

print("How temperature changes the probability distribution:\n")
for T in [0.3, 1.0, 2.0, 5.0]:
    probs = apply_temperature(teacher_logits, T)
    print(f"  T={T:.1f}: ", end="")
    for label, p in zip(labels, probs):
        print(f"{label}={p:.3f}  ", end="")
    print()

print("\n  T=0.3 → nearly one-hot (hard label)")
print("  T=2.0 → softened, reveals class relationships")
print("  T=5.0 → nearly uniform (too much — loses information)")

## 6.10 The Distillation Loss

```
Loss = α × L_hard + (1-α) × L_soft
```

| Term | What it is | What it does |
|---|---|---|
| **L_hard** | Cross-entropy with ground-truth labels | Anchors student to correct answers |
| **L_soft** | KL divergence between student and teacher distributions (at temperature T) | Transfers teacher's "dark knowledge" |
| **α** | Weight between losses (0.1-0.5) | Balances correctness vs. knowledge transfer |
| **T²** | Soft loss scaling factor | Compensates for gradient shrinking at high T |

For small models with LoRA, hard-label distillation alone gives most of the benefit. Soft labels help most with fine-grained categorization, small datasets, or when teacher >> student.

## 6.11 Save the Distilled Adapter

In [ ]:
adapter_path = "./adapter"
model.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")